# 03. 합치기 + 학습/테스트 나누기 + 최종 채점
**작성자: C**

- 6주차 1·2단계: 배포 상황 = '센서 없는 새 동네' → **자치구 단위**로 나누고 테스트는 잠금
- ⚠️ `test.csv`는 **공유 폴더가 아니라 C 개인 드라이브**에 저장

In [ ]:
# ── 공통 설정 (모든 노트북 맨 위에 그대로 두기) ──
from google.colab import drive
drive.mount('/content/drive')

BASE = '/content/drive/MyDrive/AICEE_project'   # 모두 같은 경로 (공유 폴더를 '내 드라이브에 바로가기 추가' 해야 함)
DATA = f'{BASE}/data_processed'
RESULTS = f'{BASE}/results'

import sys
sys.path.append(f'{BASE}/shared')
RANDOM_STATE = 42

USE_DUMMY = True   # ← 진짜 데이터 나오면 False로 바꾸기

In [ ]:
import pandas as pd, numpy as np
labels = pd.read_csv(f'{DATA}/labels.csv')
feats = pd.read_csv(f'{DATA}/features.csv').drop(columns=['gu', 'lat', 'lon'])
data = labels.merge(feats, on='sensor_id', how='inner')
print('합친 뒤', data.shape, '| 결측 있는 행', data.isna().any(axis=1).sum())
data = data.dropna()
assert data['sensor_id'].is_unique, '중복 센서 있음 (6주차: 중복 제거)'
print(data.shape, data['gu'].nunique(), '개 구')

In [ ]:
rng = np.random.default_rng(RANDOM_STATE)
test_gu = sorted(rng.choice(sorted(data['gu'].unique()), 5, replace=False))
print('테스트 구:', test_gu)   # 강남·강북, 한강변, 산자락이 섞였는지 확인! 치우치면 seed 바꾸고 이유 기록

In [ ]:
train = data[~data['gu'].isin(test_gu)]
test = data[data['gu'].isin(test_gu)]
train.to_csv(f'{DATA}/train.csv', index=False)
test.to_csv('/content/drive/MyDrive/test_LOCKED.csv', index=False)   # 개인 드라이브
print(train.shape, test.shape)

> 6주차 참고: 공간 데이터는 경계에 '완충 거리'를 두라고 함. 테스트 구 경계 근처 학습 센서 때문에 점수가 조금 좋게 나올 수 있음 → 한계로 기록

## 최종 채점 — 모델 4개 확정 후 **딱 한 번** (6주차 6단계)
1. 각 모델을 train 전체로 학습 → test 예측
2. RMSE + **95% 신뢰구간** (부트스트랩)
3. 두 모델 차이는 **paired bootstrap**으로: 구간이 0을 포함하면 '차이 없음'

In [ ]:
# from evaluate import bootstrap_ci, paired_bootstrap, latest_scores
# test = pd.read_csv('/content/drive/MyDrive/test_LOCKED.csv')
# preds = {}   # 예: preds['model4_XGB'] = final_xgb.fit(train[F], train[t]).predict(test[F])
# for name, p in preds.items():
#     rmse, (lo, hi) = bootstrap_ci(test['uhi_night'], p)
#     print(f'{name}: RMSE {rmse:.3f} °C, 95% CI [{lo:.3f}, {hi:.3f}]')
# print('XGB - RF 차이 구간:', paired_bootstrap(test['uhi_night'], preds['model4_XGB'], preds['model3_RF']))
# latest_scores(BASE)   # 교차검증 비교표 (n_tries = 시도 횟수도 같이 보고)